In [2]:
import pandas as pd
from pathlib import Path
import ast

In [3]:
syllable_frequencies = pd.read_csv('/home/lillianchang/Documents/MOUS_hierarchical-representations/tables/guslatho_MOUS_word_syllable_frequencies.csv',index_col=0)
syllable_frequencies.head()

,Word,n_syllables,Log10_Syllable_FREQcounts,Syllable_Zipfs,Min_Log10_FreqCount,Min_Syll_Zipf
0,ik,1,"(6.241737438532683,)","(7.468668823067624,)",6.241737,7.468669
1,je,1,"(6.249430979053402,)","(7.476362363588343,)",6.249431,7.476362
2,het,1,"(6.032457179694622,)","(7.259388564229563,)",6.032457,7.259389
3,de,1,"(6.234172950952972,)","(7.461104335487914,)",6.234173,7.461104
4,dat,1,"(6.012697215956926,)","(7.239628600491868,)",6.012697,7.239629


In [4]:
print(type(syllable_frequencies['Syllable_Zipfs'][0]))

<class 'str'>


In [5]:
syllable_frequencies['First_syllable_zipf'] = syllable_frequencies['Syllable_Zipfs'].apply(ast.literal_eval)
syllable_frequencies['First_syllable_zipf'] = syllable_frequencies['First_syllable_zipf'].str[0]
syllable_frequencies.head(200)

,Word,n_syllables,Log10_Syllable_FREQcounts,Syllable_Zipfs,Min_Log10_FreqCount,Min_Syll_Zipf,First_syllable_zipf
0,ik,1,"(6.241737438532683,)","(7.468668823067624,)",6.241737,7.468669,7.468669
1,je,1,"(6.249430979053402,)","(7.476362363588343,)",6.249431,7.476362,7.476362
2,het,1,"(6.032457179694622,)","(7.259388564229563,)",6.032457,7.259389,7.259389
3,de,1,"(6.234172950952972,)","(7.461104335487914,)",6.234173,7.461104,7.461104
4,dat,1,"(6.012697215956926,)","(7.239628600491868,)",6.012697,7.239629,7.239629
...,...,...,...,...,...,...,...
195,na,1,"(5.202455621138131,)","(6.429387005673073,)",5.202456,6.429387,6.429387
196,graag,1,"(4.415490852171469,)","(5.64242223670641,)",4.415491,5.642422,5.642422
197,toe,1,"(4.799698866883532,)","(6.026630251418473,)",4.799699,6.026630,6.026630
198,helpen,2,"(4.5310827620341625, 5.250193036344548)","(5.758014146569104, 6.477124420879489)",4.531083,5.758014,5.758014


In [6]:
export = syllable_frequencies[['Word','n_syllables','Min_Syll_Zipf','First_syllable_zipf']]
export.to_csv('/home/lillianchang/Documents/MOUS_hierarchical-representations/tables/first_syllable_frequency.csv')

In [8]:
export

,Word,n_syllables,Min_Syll_Zipf,First_syllable_zipf
0,ik,1,7.468669,7.468669
1,je,1,7.476362,7.476362
2,het,1,7.259389,7.259389
3,de,1,7.461104,7.461104
4,dat,1,7.239629,7.239629
...,...,...,...,...
278056,aagghh,1,1.527961,1.527961
278057,aagezien,3,2.745445,2.745445
278058,aaf,1,1.527961,1.527961
278059,aada's,0,NaN,NaN


In [14]:
source = Path('/media/lillianchang/MOUSnew/SynologyDrive/source/SynologyDrive')

for subject in source.rglob('sub-A*'):
    if subject.is_dir():
        # Read the transcription file
        transcription = pd.read_csv(str(subject / 'func' / f'{subject.name}_transcription.csv'))
        
        # Rename the column for consistency
        transcription.rename(columns={'Transcription': 'Word'}, inplace=True)
        transcription['Word'] = transcription['Word'].apply(lambda x: str(x).lower())
        # Merge transcription with syllable_table on the Word column
        transcription = transcription.merge(
            export[['Word', 'n_syllables', 'Min_Syll_Zipf','First_syllable_zipf']],
            on='Word',
            how='left'
        )
        
        # Rename columns for clarity
        transcription.rename(
            columns={
                'Min_Syll_Zipf': 'MinSyllZipf'
            },
            inplace=True
        )
        outpath = subject / 'func' / f'{subject.name}_first_syllable_frequency.csv'
        transcription.to_csv(str(outpath))
